In [1]:
#Cell 1 — load config and cleaned data:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from src.utils import load_config, set_seed
from src.preprocessing.splitting import split_features_labels, make_train_test_split, verify_split
from src.preprocessing.scaling import fit_scaler, apply_scaler, save_scaler

cfg = load_config()
set_seed(cfg["project"]["random_state"])

parquet_path = os.path.join(cfg["paths"]["processed_dir"], "cicids2017_ddos_cleaned.parquet")
df_clean = pd.read_parquet(parquet_path)
print("Loaded cleaned data:", df_clean.shape)

Loaded cleaned data: (223080, 69)


In [2]:
#Cell 2 — separate features/labels:
X, y = split_features_labels(
    df_clean,
    label_col=cfg["dataset"]["label_column"],
    binary_col="label_binary",
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns (first 10):", list(X.columns[:10]))
print("\nIs 'Label' in X?          :", "Label" in X.columns, "(must be False)")
print("Is 'label_binary' in X?    :", "label_binary" in X.columns, "(must be False)")

X shape: (223080, 67)
y shape: (223080,)

Feature columns (first 10): ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Is 'Label' in X?          : False (must be False)
Is 'label_binary' in X?    : False (must be False)


In [3]:
#Cell 3 — split:
X_train, X_test, y_train, y_test = make_train_test_split(X, y, cfg)

split_check = verify_split(X_train, X_test, y_train, y_test, verbose=True)

SPLIT VERIFICATION
Train samples: 178,464
Test samples:  44,616
Index overlap between train/test: 0 (MUST be 0)
----------------------------------------------------------------
Train class ratio: {0: 0.4262, 1: 0.5738}
Test class ratio:  {0: 0.4261, 1: 0.5739}


In [4]:
#Cell 4 — fit scaler on TRAIN ONLY, transform both:
scaler = fit_scaler(X_train)

X_train_scaled = apply_scaler(scaler, X_train)
X_test_scaled = apply_scaler(scaler, X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)

print("\nTrain scaled mean (should be ~0 for every column):")
print(X_train_scaled.mean().abs().max())  # max absolute mean across all columns

print("\nTest scaled mean (will NOT be exactly 0 - this is expected and correct,")
print("it proves we did not fit on test data):")
print(X_test_scaled.mean().abs().max())

X_train_scaled shape: (178464, 67)
X_test_scaled shape : (44616, 67)

Train scaled mean (should be ~0 for every column):
3.20585110069276e-16

Test scaled mean (will NOT be exactly 0 - this is expected and correct,
it proves we did not fit on test data):
0.011821849656277132


In [5]:
#Cell 5 — save everything:
import joblib

processed_dir = cfg["paths"]["processed_dir"]
models_dir = cfg["paths"]["models_dir"]

# Save raw (unscaled) splits - useful for tree models later
X_train.to_parquet(os.path.join(processed_dir, "X_train_raw.parquet"))
X_test.to_parquet(os.path.join(processed_dir, "X_test_raw.parquet"))

# Save scaled splits - useful for Logistic Regression
X_train_scaled.to_parquet(os.path.join(processed_dir, "X_train_scaled.parquet"))
X_test_scaled.to_parquet(os.path.join(processed_dir, "X_test_scaled.parquet"))

# Save labels
y_train.to_frame(name="label_binary").to_parquet(os.path.join(processed_dir, "y_train.parquet"))
y_test.to_frame(name="label_binary").to_parquet(os.path.join(processed_dir, "y_test.parquet"))

# Save the fitted scaler
save_scaler(scaler, os.path.join(models_dir, "standard_scaler.joblib"))

print("\nAll train/test artifacts saved to:", processed_dir)
print("Scaler saved to:", models_dir)

Scaler saved: d:\MSc_Project\Network_IDS_Framework\results/models\standard_scaler.joblib

All train/test artifacts saved to: d:\MSc_Project\Network_IDS_Framework\results/processed
Scaler saved to: d:\MSc_Project\Network_IDS_Framework\results/models
